<a href="https://colab.research.google.com/github/sri-wahyuni10/Inter-flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Logistic Regression -> Random Forest.**

The question for this lane is a yes/no question with an observed label: is a piece of content
underperforming on CTR relative to its peers in the same impression-volume bucket (this extends
the "quick-win" volume signal that was CONFIRMED in Week 4, Signal 1 — higher impression buckets
have higher average clicks, so a "fair" CTR expectation depends on the bucket a piece of content
sits in). That makes this a "yes/no with an observed label" problem, so per the toolkit I start
with Logistic Regression — simple, and every coefficient can be explained to a non-technical
stakeholder — then check whether Random Forest actually earns its extra complexity on the same
metric before trusting it.

The label is NOT the baseline's own `action_score` formula. If it were, the model would just be
memorizing my own rule (and any feature that includes clicks/CTR would trivially "predict" it —
that's leakage, not learning). Instead: `y=1` if a content's CTR is below the median CTR of its
own impression bucket, **with that median computed only from the train split** — a real,
independently-observed outcome, not a copy of the baseline's math.

Features used are all pre-click signals only: `avg_position`, `n_days`, `total_impressions`, and
GA4/session/AI-referral engagement columns. `total_clicks`/`ctr` are deliberately excluded from
the feature set because they are the source of the label itself.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

import gc, os
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Same warehouse connection as w04 ---
con = duckdb.connect()
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Make sure you have set the Secrets 'HF_TOKEN' in Google Colab!")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET max_memory='4GB';")
con.execute("SET threads=2;")
con.execute(f"""
    CREATE SECRET http_auth (
        TYPE http,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Bearer {hf_token}'}}
    );
""")

dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

# --- Aggregate one row per content_hash_id. Only pre-click features + what's needed to build the label. ---
q_features = f"""
SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)                as client_hash_id,
    SUM(gsc_impressions)                     as total_impressions,
    SUM(gsc_clicks)                          as total_clicks,   -- kept ONLY to build the label, not a feature
    AVG(gsc_avg_position)                    as avg_position,
    COUNT(*)                                 as n_days,
    SUM(ga4_pageviews)                       as ga4_pageviews,
    SUM(ga4_sessions)                        as ga4_sessions,
    SUM(ga4_users)                           as ga4_users,
    SUM(ga4_engaged_sessions)                as ga4_engaged_sessions,
    SUM(ga4_total_engagement_sec)            as ga4_total_engagement_sec,
    SUM(sessions_organic)                    as sessions_organic,
    SUM(sessions_direct)                     as sessions_direct,
    SUM(sessions_referral)                   as sessions_referral,
    SUM(sessions_social)                     as sessions_social,
    SUM(sessions_paid)                       as sessions_paid,
    SUM(sessions_ai)                         as sessions_ai,
    SUM(ai_chatgpt)                          as ai_chatgpt,
    SUM(ai_perplexity)                       as ai_perplexity,
    SUM(ai_gemini)                           as ai_gemini,
    SUM(ai_copilot)                          as ai_copilot,
    SUM(ai_claude)                           as ai_claude,
    SUM(ai_meta)                             as ai_meta,
    SUM(ai_other)                            as ai_other,
    SUM(scroll_events)                       as scroll_events
FROM read_parquet('{dataset_url}')
WHERE gsc_data_available = true
GROUP BY 1
"""
df = con.execute(q_features).df()
df["ctr"] = np.where(df["total_impressions"] > 0, df["total_clicks"] / df["total_impressions"], 0)
df = df[df["total_impressions"] >= 10].reset_index(drop=True)  # same filter as the w04 baseline

# Same impression buckets as Week-4 Signal 1 -- this is what the label will be relative to.
imp_bins = [0, 10, 100, 1000, np.inf]
imp_labels = ["1.Low", "2.Medium", "3.High", "4.Critical"]
df["imp_bucket"] = pd.cut(df["total_impressions"], bins=imp_bins, labels=imp_labels, right=False)

# --- Handle NaN BEFORE feature_cols/split are defined, so every downstream cell inherits clean data ---
# SUM() in DuckDB returns NULL (not 0) for a content item that has zero GA4-connected rows at all --
# i.e. NaN here means "this client never had GA4 wired up for this content", not "random missing value".
ga4_session_ai_cols = [
    "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid", "sessions_ai",
    "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
    "scroll_events",
]
n_nan_before = df[ga4_session_ai_cols].isna().sum().sum()
df[ga4_session_ai_cols] = df[ga4_session_ai_cols].fillna(0)

# avg_position: NaN means position genuinely wasn't measured for that content (a data gap), not "position 0".
# Fill with the worst plausible value so the model never mistakes "unmeasured" for "ranks well".
df["avg_position"] = df["avg_position"].fillna(df["avg_position"].max())

# Explicit flag: was GA4 connected at all for this content, or is it structurally absent?
# Needed so the model can tell "0 because no GA4" apart from "0 because genuinely quiet" --
# those mean very different things and collapsing them would hide a client-setup artifact as a real signal.
q_ga4_flag = f"""
SELECT
    content_hash_id,
    MAX(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) as has_ga4_data
FROM read_parquet('{dataset_url}')
GROUP BY 1
"""
df_ga4_flag = con.execute(q_ga4_flag).df()
df = df.merge(df_ga4_flag, on="content_hash_id", how="left")

print(f"NaN found across GA4/session/AI columns before fix: {n_nan_before:,}")
print(f"Content with no GA4 connected at all: {(df['has_ga4_data']==0).sum():,} of {len(df):,} "
      f"({(df['has_ga4_data']==0).mean():.1%})")
print(f"Remaining NaN in df after fix: {df.isna().sum().sum()} (must be 0)")
assert df[ga4_session_ai_cols + ['avg_position']].isna().sum().sum() == 0, "NaN still present -- check upstream."

feature_cols = [
    "avg_position", "n_days", "total_impressions", "has_ga4_data",
    "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions", "ga4_total_engagement_sec",
    "sessions_organic", "sessions_direct", "sessions_referral", "sessions_social", "sessions_paid", "sessions_ai",
    "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot", "ai_claude", "ai_meta", "ai_other",
    "scroll_events",
]

print(f"\nTotal unique content: {len(df):,}  |  Total unique clients: {df['client_hash_id'].nunique():,}")
print(f"\nNumber of features used (all pre-click, no clicks/ctr): {len(feature_cols)}")
print(feature_cols)
print("\nImpression bucket distribution (whole dataset, before split):")
print(df["imp_bucket"].value_counts().sort_index())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NaN found across GA4/session/AI columns before fix: 749,436
Content with no GA4 connected at all: 77,091 of 143,206 (53.8%)
Remaining NaN in df after fix: 0 (must be 0)

Total unique content: 143,206  |  Total unique clients: 45

Number of features used (all pre-click, no clicks/ctr): 23
['avg_position', 'n_days', 'total_impressions', 'has_ga4_data', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Impression bucket distribution (whole dataset, before split):
imp_bucket
1.Low             0
2.Medium      41765
3.High        56383
4.Critical    45058
Name: count, dtype: int64


## 2. Split design

**Split: grouped by `client_hash_id`, 80/20.**

The data is a single-month snapshot (March 2026), so a time-based split doesn't apply here. The
real leakage risk is a **client** leaking across the split: content from the same client tends to
share title/meta style and engagement patterns, so if one client's content sits in both train and
test, the model could "cheat" by memorizing that client's style instead of learning a general
signal. `GroupShuffleSplit` on `client_hash_id` guarantees no client appears on both sides (checked
below: overlap must be 0).

The label itself is built from a threshold computed **only on the train rows** (median CTR per
impression bucket), then applied to both train and test — so the definition of "underperforming"
never gets to see the test set.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, groups=df["client_hash_id"]))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

overlap = set(df_train["client_hash_id"]) & set(df_test["client_hash_id"])
print(f"Train: {len(df_train):,} content items from {df_train['client_hash_id'].nunique():,} clients")
print(f"Test : {len(df_test):,} content items from {df_test['client_hash_id'].nunique():,} clients")
print(f"Clients leaking into both splits: {len(overlap)} (must be 0)")
assert len(overlap) == 0, "Group split failed -- a client leaked across train/test."

# Label threshold computed from TRAIN ONLY, then applied to both train and test.
bucket_median_ctr = df_train.groupby("imp_bucket")["ctr"].median()
print("\nMedian CTR per bucket (computed from TRAIN only, used as the threshold):")
print(bucket_median_ctr)

def make_label(frame):
    thr = frame["imp_bucket"].map(bucket_median_ctr)
    return (frame["ctr"] < thr).astype(int)

df_train["y"] = make_label(df_train)
df_test["y"]  = make_label(df_test)

print(f"\nBase rate train: {df_train['y'].mean():.3f}")
print(f"Base rate test : {df_test['y'].mean():.3f}")

X_train, y_train = df_train[feature_cols], df_train["y"]
X_test,  y_test  = df_test[feature_cols],  df_test["y"]


Train: 112,102 content items from 36 clients
Test : 31,104 content items from 9 clients
Clients leaking into both splits: 0 (must be 0)

Median CTR per bucket (computed from TRAIN only, used as the threshold):
imp_bucket
1.Low              NaN
2.Medium      0.000000
3.High        0.000000
4.Critical    0.001906
Name: ctr, dtype: float64

Base rate train: 0.159
Base rate test : 0.141


/tmp/ipykernel_304/4077316951.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_median_ctr = df_train.groupby("imp_bucket")["ctr"].median()


## 3. Train + compare vs my baseline

The table below reports precision@50, precision@100, and ROC-AUC for the Week-4 baseline
(ranked by `action_score`) and both models, **on the same grouped test split**, plus the test
base rate for context -- if a model's precision@K is close to the base rate, it is doing no
better than guessing at that K.

*(Fill in the actual numbers/conclusion here after running the cell below -- e.g. "Random Forest
wins at precision@50 but loses at precision@100 against the baseline -- both are worth reporting,
that IS the finding.")*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

# --- Model 1: Logistic Regression ---
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")),
])
pipe_lr.fit(X_train, y_train)
proba_lr = pipe_lr.predict_proba(X_test)[:, 1]

# --- Model 2: Random Forest ---
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20,
    random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
)
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

# --- Bring in the Week-4 baseline's score for the SAME test-set content ---
baseline_csv_path = "/content/baseline_action_score (1).csv"  # adjust path if needed
df_baseline = pd.read_csv(baseline_csv_path)
df_test_b = df_test.merge(
    df_baseline[["content_hash_id", "action_score"]], on="content_hash_id", how="left"
)
df_test_b["action_score"] = df_test_b["action_score"].fillna(0)  # content that didn't clear baseline's filter -> score 0

Ks = [50, 100]
rows = []
for k in Ks:
    rows.append({"method": "Baseline (w04 rule)", "K": k,
                 "precision@K": precision_at_k(df_test_b["y"], df_test_b["action_score"].values, k)})
    rows.append({"method": "Logistic Regression", "K": k,
                 "precision@K": precision_at_k(y_test, proba_lr, k)})
    rows.append({"method": "Random Forest", "K": k,
                 "precision@K": precision_at_k(y_test, proba_rf, k)})

comparison_table = pd.DataFrame(rows).pivot(index="method", columns="K", values="precision@K")
comparison_table.columns = [f"precision@{k}" for k in comparison_table.columns]
comparison_table["ROC-AUC"] = [
    roc_auc_score(df_test_b["y"], df_test_b["action_score"]),
    roc_auc_score(y_test, proba_lr),
    roc_auc_score(y_test, proba_rf),
]
comparison_table["base_rate_test"] = df_test["y"].mean()

print("="*72)
print("MODEL vs BASELINE -- same test split, same metric")
print("="*72)
display(comparison_table)


MODEL vs BASELINE -- same test split, same metric


,precision@50,precision@100,ROC-AUC,base_rate_test
method,,,,
Baseline (w04 rule),0.44,0.37,0.897638,0.140721
Logistic Regression,0.68,0.69,0.862799,0.140721
Random Forest,0.98,0.95,0.968570,0.140721


## 4. Errors and interpretation

*Fill in after running the cell below: which impression bucket does the model get wrong most
often, what does permutation importance say it leans on (and does that make sense, or look
suspiciously perfect / leak-like), and what do the 3 concrete wrong cases have in common.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

best_model_name = comparison_table["precision@50"].idxmax()
print(f"Best model by precision@50: {best_model_name}\n")

# --- Permutation importance for Random Forest ---
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print("="*72)
print("TOP 5 FEATURE IMPORTANCE (permutation, Random Forest)")
print("="*72)
display(imp_df.head(5))
print("\nSanity check: does the top feature make sense, or is it 'suspiciously perfect' (leakage)?")
print("-> avg_position dominating is expected (SERP position generally correlates with CTR).")
print("-> If a GA4/session feature suddenly dominates far above avg_position, be suspicious --")
print("   it may be a hidden proxy for the actual clicks the label is built from.")

# --- Error breakdown by impression bucket ---
df_test_err = df_test.copy()
df_test_err["proba_rf"] = proba_rf
df_test_err["pred_rf"] = (proba_rf >= 0.5).astype(int)

print("\n" + "="*72)
print("ERROR BREAKDOWN by impression bucket")
print("="*72)
err_by_bucket = df_test_err.groupby("imp_bucket").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": (g["pred_rf"] == g["y"]).mean(),
        "false_positive_rate": ((g["pred_rf"] == 1) & (g["y"] == 0)).sum() / max((g["y"] == 0).sum(), 1),
        "false_negative_rate": ((g["pred_rf"] == 0) & (g["y"] == 1)).sum() / max((g["y"] == 1).sum(), 1),
    })
)
display(err_by_bucket)

print("\n" + "="*72)
print("3 CONCRETE WRONG CASES (false negatives -- model said fine, actually underperforming)")
print("="*72)
mask_fn = (df_test_err["pred_rf"] == 0) & (df_test_err["y"] == 1)
wrong_fn = df_test_err[mask_fn].sample(n=min(3, mask_fn.sum()), random_state=RANDOM_STATE)
for _, r in wrong_fn.iterrows():
    print(f"- content_id={r['content_hash_id']}  avg_position={r['avg_position']:.1f}  "
          f"ctr={r['ctr']:.4f}  model_proba={r['proba_rf']:.3f}")
    print(f"  Likely hard because: its features look normal, but CTR is still low --")
    print(f"  something outside these features (query intent, actual title text) probably explains it.\n")


Best model by precision@50: Random Forest

TOP 5 FEATURE IMPORTANCE (permutation, Random Forest)


,feature,importance_mean,importance_std
2,total_impressions,0.232941,0.001623
9,sessions_organic,0.046994,0.000527
4,ga4_pageviews,0.014519,0.000572
0,avg_position,0.014368,0.000686
6,ga4_users,0.010076,0.000373



Sanity check: does the top feature make sense, or is it 'suspiciously perfect' (leakage)?
-> avg_position dominating is expected (SERP position generally correlates with CTR).
-> If a GA4/session feature suddenly dominates far above avg_position, be suspicious --
   it may be a hidden proxy for the actual clicks the label is built from.

ERROR BREAKDOWN by impression bucket


/tmp/ipykernel_304/137413877.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  err_by_bucket = df_test_err.groupby("imp_bucket").apply(
/tmp/ipykernel_304/137413877.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  err_by_bucket = df_test_err.groupby("imp_bucket").apply(


,n,accuracy,false_positive_rate,false_negative_rate
imp_bucket,,,,
1.Low,0.0,NaN,0.000000,0.000000
2.Medium,7608.0,1.000000,0.000000,0.000000
3.High,14123.0,1.000000,0.000000,0.000000
4.Critical,9373.0,0.648565,0.644315,0.017135



3 CONCRETE WRONG CASES (false negatives -- model said fine, actually underperforming)
- content_id=content_03434e988f11ac5c  avg_position=4.3  ctr=0.0013  model_proba=0.500
  Likely hard because: its features look normal, but CTR is still low --
  something outside these features (query intent, actual title text) probably explains it.

- content_id=content_be61feead738768d  avg_position=3.6  ctr=0.0017  model_proba=0.400
  Likely hard because: its features look normal, but CTR is still low --
  something outside these features (query intent, actual title text) probably explains it.

- content_id=content_e486f55b93e8202f  avg_position=5.2  ctr=0.0011  model_proba=0.491
  Likely hard because: its features look normal, but CTR is still low --
  something outside these features (query intent, actual title text) probably explains it.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.